# 07 — Exploratory biological annotation of spatial domains

**Input:** `results/adata/06_markers.h5ad`  
**Output:** `results/adata/07_annotated.h5ad`, annotation evidence tables and figures.

This notebook scores **curated candidate tissue programs** against each Leiden cluster. It does **not** assign single-cell identities: Visium HD 56 µm bins can contain multiple cells. Scores are relative to other clusters, not calibrated probabilities. No QC filtering or histology image registration has been verified. Gene sets are **illustrative hypotheses** requiring review against femoral-head-specific literature and histology.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "results/adata/06_markers.h5ad").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run Step 06 first: results/adata/06_markers.h5ad not found")
INPUT = ROOT / "results/adata/06_markers.h5ad"
OUTPUT = ROOT / "results/adata/07_annotated.h5ad"
FIG = ROOT / "results/figures"
TAB = ROOT / "results/tables"
for p in (OUTPUT.parent, FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED)
print("Python:", sys.executable)
print("Scanpy:", version("scanpy"), "AnnData:", version("anndata"))
print("Project root:", ROOT)

## 1. Load and validate

Keep the original Leiden IDs and all expression/spatial graphs. All subsequent annotation is **provisional**.

In [ ]:
adata = sc.read_h5ad(INPUT)
KEY = "rank_genes_leiden_wilcoxon"
assert "leiden" in adata.obs and "spatial" in adata.obsm
assert KEY in adata.uns, "Run marker analysis (Step 06) first"
assert "counts" in adata.layers and sparse.issparse(adata.layers["counts"])
assert "spatial_connectivities" in adata.obsp and "connectivities" in adata.obsp
assert "log1p" in adata.uns
adata.obs["leiden"] = adata.obs["leiden"].astype("category")
clusters = list(adata.obs["leiden"].cat.categories)
print("AnnData:", adata.shape, "Leiden clusters:", len(clusters))

## 2. Candidate biological programs and gene coverage

These are **candidate signatures**, not validated cell-type classifiers. For example, collagen I can be expressed in multiple mesenchymal compartments; inflammatory and vascular signals may reflect mixed bins. Missing genes are reported and excluded, never treated as evidence of absence.

In [ ]:
PROGRAMS = {
    "Cartilage / chondrocyte-like": ["COL2A1", "ACAN", "SOX9", "COMP", "COL9A1", "COL11A1"],
    "Bone matrix / osteogenic-like": ["COL1A1", "COL1A2", "SPP1", "IBSP", "BGLAP", "SP7"],
    "Fibroblast / stromal-like": ["DCN", "LUM", "COL3A1", "PDGFRA", "THY1", "FBLN1"],
    "Vascular endothelial-like": ["PECAM1", "VWF", "KDR", "EMCN", "CDH5", "ESAM"],
    "Perivascular / smooth-muscle-like": ["ACTA2", "TAGLN", "MYH11", "RGS5", "PDGFRB", "MCAM"],
    "Myeloid / macrophage-like": ["LST1", "TYROBP", "C1QA", "C1QB", "CD68", "CSF1R"],
    "Lymphoid-like": ["PTPRC", "CD3D", "CD3E", "TRAC", "MS4A1", "CD79A"],
    "Adipocyte-like": ["ADIPOQ", "PLIN1", "FABP4", "LPL", "LEP", "CIDEC"],
    "Erythroid-like": ["HBB", "HBA1", "HBA2", "ALAS2", "GYPA", "SLC4A1"],
}
coverage_rows = []
found = {}
for program, genes in PROGRAMS.items():
    present = [g for g in genes if g in adata.var_names]
    missing = [g for g in genes if g not in adata.var_names]
    found[program] = present
    coverage_rows.append({"program": program, "n_reference_genes": len(genes),
                          "n_detected_genes": len(present),
                          "coverage_fraction": len(present) / len(genes),
                          "present_genes": ";".join(present),
                          "missing_genes": ";".join(missing)})
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(TAB / "07_program_gene_coverage.csv", index=False)
display(coverage)
assert any(len(g) >= 2 for g in found.values()), "Too few candidate genes in the dataset"

## 3. Score candidate programs by cluster

For each program, calculate the mean **log1p-normalized expression** of available genes per bin, then average by cluster. Convert each program's cluster means into a z-score **across clusters**. These are descriptive relative scores; they do not control for gene-set size, bin depth, or spatial dependence. Programs with fewer than two detected genes are not scored.

In [ ]:
valid = {name: genes for name, genes in found.items() if len(genes) >= 2}
program_names = list(valid)
# Subset only the small marker union to avoid densifying the 14,400 × 18,085 matrix.
union = list(dict.fromkeys(g for genes in valid.values() for g in genes))
sub = adata[:, union].X
small = sub.toarray() if sparse.issparse(sub) else np.asarray(sub)
idx = {gene: i for i, gene in enumerate(union)}
per_bin = pd.DataFrame(index=adata.obs_names)
for name, genes in valid.items():
    per_bin[name] = small[:, [idx[g] for g in genes]].mean(axis=1)
per_bin["leiden"] = adata.obs["leiden"].astype(str).to_numpy()
means = per_bin.groupby("leiden", sort=False)[program_names].mean()
means = means.reindex([str(c) for c in clusters])
std = means.std(axis=0, ddof=0)
z = (means - means.mean(axis=0)) / std.replace(0, np.nan)
z = z.fillna(0.0)
means.to_csv(TAB / "07_cluster_program_mean_expression.csv", index_label="leiden")
z.to_csv(TAB / "07_cluster_program_zscores.csv", index_label="leiden")
print("Programs scored:", len(program_names), "Clusters:", len(means))
display(z.round(2).head())

## 4. Evidence from Step 06 marker rankings

Count how many signature genes are among each cluster's **top 100 positive marker candidates**, using the same exploratory Wilcoxon results as Step 06. Marker ranks and program scores are related evidence, **not independent validation**.

In [ ]:
marker_rows = []
for c in clusters:
    df = sc.get.rank_genes_groups_df(adata, group=str(c), key=KEY)
    df = df.loc[df["logfoldchanges"].fillna(-np.inf) > 0].sort_values("scores", ascending=False)
    top100 = set(df.head(100)["names"].astype(str))
    for name, genes in valid.items():
        hits = [g for g in genes if g in top100]
        marker_rows.append({"leiden": str(c), "program": name,
                            "n_signature_genes": len(genes),
                            "n_top100_marker_hits": len(hits),
                            "top100_marker_hits": ";".join(hits)})
evidence = pd.DataFrame(marker_rows)
evidence.to_csv(TAB / "07_program_marker_evidence.csv", index=False)
print("Evidence rows:", len(evidence))

## 5. Conservative candidate labels and review flags

A label is proposed only when a program has a relative z-score ≥ 1, at least **two** signature genes in the cluster's top 100 positive marker list, and a margin ≥ 0.5 z-score over the runner-up. Otherwise label **Unresolved / mixed**. These thresholds are **heuristics**, not statistical confidence intervals. Even a proposed label must be manually reviewed.

In [ ]:
MIN_Z = 1.0
MIN_HITS = 2
MIN_MARGIN = 0.5
annotation_rows = []
for c in z.index:
    ordered = z.loc[c].sort_values(ascending=False)
    best = str(ordered.index[0]); best_z = float(ordered.iloc[0])
    second_z = float(ordered.iloc[1]) if len(ordered) > 1 else float("nan")
    margin = best_z - second_z if np.isfinite(second_z) else 0.0
    match = evidence.loc[(evidence["leiden"] == c) & (evidence["program"] == best)].iloc[0]
    hits = int(match["n_top100_marker_hits"])
    proposed = best_z >= MIN_Z and hits >= MIN_HITS and margin >= MIN_MARGIN
    annotation_rows.append({"leiden": c, "n_bins": int((adata.obs["leiden"].astype(str) == c).sum()),
                            "candidate_program": best,
                            "best_program_zscore": best_z,
                            "runner_up_program": str(ordered.index[1]) if len(ordered) > 1 else "",
                            "zscore_margin": margin,
                            "n_top100_marker_hits": hits,
                            "supporting_markers": str(match["top100_marker_hits"]),
                            "provisional_label": best if proposed else "Unresolved / mixed",
                            "review_status": "Manual review required",
                            "interpretation": "Spatial-bin program; not a cell-type call"})
annotation = pd.DataFrame(annotation_rows)
annotation.to_csv(TAB / "07_provisional_cluster_annotations.csv", index=False)
print("Proposed candidates:", (annotation["provisional_label"] != "Unresolved / mixed").sum(),
      "/", len(annotation))
display(annotation[["leiden", "candidate_program", "best_program_zscore",
                    "n_top100_marker_hits", "provisional_label"]])

## 6. Visualize program patterns and spatial distribution

The heatmap displays relative program scores; the map displays **Leiden bins colored by provisional program**. It is **coordinate-only**, without a tissue-image overlay.

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(7, len(z) * 0.28)))
im = ax.imshow(z.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-2, vmax=2)
ax.set_xticks(range(len(z.columns)), z.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(z.index)), z.index, fontsize=8)
ax.set(xlabel="Candidate tissue program", ylabel="Leiden cluster",
       title="Exploratory program scores (z across clusters; clipped ±2)")
fig.colorbar(im, ax=ax, label="Relative z-score", shrink=0.75)
fig.tight_layout()
fig.savefig(FIG / "07_program_score_heatmap.png", dpi=170, bbox_inches="tight")
plt.show()

mapping = dict(zip(annotation["leiden"], annotation["provisional_label"]))
adata.obs["provisional_domain_program"] = pd.Categorical(
    adata.obs["leiden"].astype(str).map(mapping).fillna("Unresolved / mixed"))
xy = np.asarray(adata.obsm["spatial"])
labels = adata.obs["provisional_domain_program"].astype(str).to_numpy()
unique_labels = list(pd.unique(labels))
colors = plt.get_cmap("tab20", max(1, len(unique_labels)))
fig, ax = plt.subplots(figsize=(10, 8))
for i, label in enumerate(unique_labels):
    mask = labels == label
    ax.scatter(xy[mask, 0], xy[mask, 1], s=3, linewidths=0,
               color="0.7" if label == "Unresolved / mixed" else colors(i),
               label=f"{label} ({mask.sum():,})", rasterized=True)
ax.invert_yaxis()
ax.set_aspect("equal", adjustable="box")
ax.set(xlabel="Fullres x (pixels)", ylabel="Fullres y (pixels)",
       title="Provisional spatial-bin programs (no image overlay)")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, markerscale=4)
fig.tight_layout()
fig.savefig(FIG / "07_provisional_program_spatial.png", dpi=170, bbox_inches="tight")
plt.show()

## 7. Save and reload validation

The original Leiden clusters remain unchanged. No cell-type label is asserted.

In [ ]:
adata.uns["step07_settings"] = {
    "annotation_type": "exploratory_spatial_bin_program",
    "score": "mean_log1p_expression_per_program_then_cluster_zscore",
    "min_program_zscore": float(MIN_Z),
    "min_top100_marker_hits": int(MIN_HITS),
    "min_zscore_margin": float(MIN_MARGIN),
    "manual_review_required": True,
    "not_cell_type_labels": True,
    "qc_filters_applied": False,
    "image_registration_verified": False,
}
adata.write_h5ad(OUTPUT, compression="gzip")
check = sc.read_h5ad(OUTPUT)
assert check.shape == adata.shape
assert check.obs["leiden"].equals(adata.obs["leiden"])
assert "provisional_domain_program" in check.obs
assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
assert "spatial_connectivities" in check.obsp and "connectivities" in check.obsp
assert "counts" in check.layers and sparse.issparse(check.layers["counts"])
assert KEY in check.uns
assert (TAB / "07_provisional_cluster_annotations.csv").exists()
assert (FIG / "07_program_score_heatmap.png").exists()
print("PASS — 07_biological_annotation completed")
print("AnnData:", check.shape)
print("Leiden clusters:", check.obs["leiden"].nunique())
print("Candidate programs:", len(program_names))
print("Provisional candidate clusters:",
      int((annotation["provisional_label"] != "Unresolved / mixed").sum()))
print("Output:", OUTPUT.relative_to(ROOT))

### Next

**Manual review is required** before biological naming: examine `07_provisional_cluster_annotations.csv`, marker evidence, known femoral-head anatomy, potential batch/technical effects, and the absence of QC filtering. For reproducibility, the next steps can be `08_robustness_checks.ipynb`, an optional Tangram notebook **only if a suitable scRNA-seq reference is available**, then automated execution, README, and finally `environment.yml`.